In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("datasets/train.csv")
M, N = df.shape
df.head(100)

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,fit,9.00,87.8,20.85,2714.0,12154.0,54.5,1.85,balanced,low,good,active,no,male
96,96,unhealthy,5.34,69.2,24.66,2843.0,NaN,31.7,1.75,non-veg,high,average,active,yes,female
97,97,at-risk,7.78,67.8,24.16,1827.0,8097.0,47.5,NaN,balanced,high,poor,moderate,no,female
98,98,at-risk,8.42,64.0,24.14,2440.0,14273.0,63.6,1.43,non-veg,high,good,active,no,other


### Анализ количества и природы пропусков

In [3]:
(df["gender"] == "other").sum()

np.int64(206943)

In [4]:
# сначала найдём все потенциально невалидные даанные
borders = {"sleep_duration": [0.0, 24.0], "heart_rate": [0.0, 220.0], "bmi": [0.0, 100],
           "calorie_expenditure": [0.0, np.inf], "step_count": [0.0, np.inf], 
           "exercise_duration": [0.0, 1440.0], "water_intake": [0.0, np.inf],
           "diet_type": ("veg", "non-veg", "balanced"), "stress_level": ("low", "high", "medium"),
           "sleep_quality": ('average', 'poor', 'good'), "physical_activity_level": ('sedentary', 'moderate', 'active'),
           "smoking_alcohol": ('yes', 'occasional', 'no'), "gender": ('female', 'other', 'male')}

for col_name in df.columns:
    if col_name not in borders:
        continue
    else:
        try:
            if type(borders[col_name]) is tuple:
                mask = ~(df[col_name].isin(borders[col_name]) | df[col_name].isna())
            elif type(borders[col_name]) is list:
                mask = ~(((df[col_name] >= borders[col_name][0]) & (df[col_name] <= borders[col_name][1])) | df[col_name].isna())
            else:
                raise AttributeError("Нестандартный тип данных из словаря borders")
        except AttributeError as ae:
            print(f"Словарь borders повреждён: {ae}")
        print(f"Столбец {col_name}: {mask.sum()}")
        
            

Столбец sleep_duration: 0
Столбец heart_rate: 0
Столбец bmi: 0
Столбец calorie_expenditure: 0
Столбец step_count: 0
Столбец exercise_duration: 0
Столбец water_intake: 0
Столбец diet_type: 0
Столбец stress_level: 0
Столбец sleep_quality: 0
Столбец physical_activity_level: 0
Столбец smoking_alcohol: 0
Столбец gender: 0


### Вывод
Данные лежат в валидных интервалах

In [5]:
df.isnull().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [6]:
missing_per_row = df.isnull().sum(axis=1)
missing_counts = [[num, np.round(num/M, 2)] for num in missing_per_row.value_counts().sort_index()]
print("Количество строк с определенным числом пропусков (кол-во, процент от всего df):")
print(*missing_counts, sep="\n")

Количество строк с определенным числом пропусков (кол-во, процент от всего df):
[349623, np.float64(0.51)]
[248134, np.float64(0.36)]
[77311, np.float64(0.11)]
[13445, np.float64(0.02)]
[1479, np.float64(0.0)]
[87, np.float64(0.0)]
[9, np.float64(0.0)]


In [7]:
# так как общий процент сэмплов с 3+ пропусками <= 0.2, то откинем их
df = df[~(df.isnull().sum(axis=1) >= 3)]

Дальше надо решить проблему обработки категориальных признаков. Воспользуемся One Hot Encoding

In [8]:
from sklearn.preprocessing import OneHotEncoder

columns_categor = ["diet_type", "stress_level", "sleep_quality", "physical_activity_level", "smoking_alcohol", "gender"]
df_numerical = df.drop(columns=columns_categor)
df_categorical = df[columns_categor].fillna("Missing")

encoder = OneHotEncoder(sparse_output=False)
df_categorical_encoded = encoder.fit_transform(df_categorical)
df_categorical = pd.DataFrame(
    df_categorical_encoded,
    columns=encoder.get_feature_names_out(columns_categor)
)

df = pd.concat([df_numerical, df_categorical], axis=1)

#### Разделим данные

In [9]:
from sklearn.utils import shuffle
import numpy as np
RS = 42


df = df.drop(["id"], axis=1)
df_shuffled = shuffle(df, random_state=RS)

X = df.drop(["health_condition"], axis=1)
Y = df["health_condition"]

#### На 3 выборки: train(70%), validate(20%), test(10%)

In [10]:
X_train, X_val, X_test = np.split(X, [int(0.7*M), int(0.9*M)])
Y_train, Y_val, Y_test = np.split(Y, [int(0.7*M), int(0.9*M)])

#### Пропуски в численных данных заполним просто медианами

In [12]:
from sklearn.impute import SimpleImputer

Imputer_train = SimpleImputer(strategy='median')
Imputer_train.fit_transform(X_train)
Imputer_val = SimpleImputer(strategy='median')
Imputer_val.fit_transform(X_val)
Imputer_test = SimpleImputer(strategy='median')
Imputer_test.fit_transform(X_test)

array([[ 7.2 , 66.9 , 27.51, ...,  1.  ,  0.  ,  0.  ],
       [ 7.03, 94.3 , 26.54, ...,  0.  ,  1.  ,  0.  ],
       [ 6.51, 65.1 , 21.85, ...,  0.  ,  0.  ,  1.  ],
       ...,
       [ 6.97, 75.1 , 22.99, ...,  1.  ,  0.  ,  0.  ],
       [ 6.97, 75.1 , 22.99, ...,  0.  ,  1.  ,  0.  ],
       [ 6.97, 75.1 , 22.99, ...,  1.  ,  0.  ,  0.  ]], shape=(68703, 31))